## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from modelbase.ode import Model, Simulator, mca
import pandas as pd
import seaborn as sns
import pickle
from model import (
    get_enzyme_costs,
    get_model,
    get_enzyme_concentrations,
    get_enzyme_parameters,
    get_proteomics_results,
    calculate_min_kcats,
)
from modelbase.ode.simulators import _Simulate
from parallel import parallelise
from pathlib import Path
from sklearn.pipeline import Pipeline
from typing import Literal, cast
from plot import savefig
from functools import partial
from typing import TypeVar


T = TypeVar("T")


def unwrap(x: T | None) -> T:
    if x is None:
        raise ValueError("Unexpected None")
    return x


CELL_VOLUME: float = 2e-15  # FIXME: explain values
PROTEIN_CONTENT: float = 0.445  # FIXME: explain values
MODE = "blast"
CHLA = (
    7.5 / 893.509 / (105482164.18012363 * 1000)  # FIXME: explain values
)  # mg/OD/L -> mmol/OD/L / Cells/LLOD --> mmol/cell
CAROT = 2 / 581.5565 / (105482164.18012363 * 1000)  # FIXME: explain values
SQUALENE = 0.5 / 410.73 / (105482164.18012363 * 1000)  # FIXME: explain values
DOUBLING_TIME: float = 300.0  # FIXME: explain values
DAY_IN_MIN = 60 * 24

# Load data
DATA = Path("data")

with open(DATA / "rate_regression_model_biomass.pickle", "rb") as f:
    BIOMASS_REGRESSION_MODEL = pickle.load(f)

OCP_ABSORPTION = pd.read_csv(DATA / "OCP_absorption.csv", index_col=0)
PS_COMP = pd.read_csv(DATA / "photosystem_composition.csv", index_col=0)
MOLAR_MASSES = cast(
    pd.Series, pd.read_csv(DATA / "molar_masses.csv", index_col=0).squeeze(1)
)

PIGMENTS_FUENTE = cast(
    pd.Series,
    pd.read_csv(
        DATA / "pigment_concentrations.csv",
        index_col=0,
    ).squeeze(1),
)

ABS_COEF = pd.concat(
    {
        pig: pd.read_csv(DATA / f"per_pigment/{pig}.csv", index_col=0)
        for pig in PIGMENTS_FUENTE.index
    },
    axis=1,
)

ENZYME_COSTS = get_enzyme_costs(
    kugler_model_csv=DATA / "Amino_acid_costs_Kuglermodel.csv",
    aa_costs_csv=DATA / "Amino_acid_costs_arnold.csv",
)

ABUNDANCES, DETECTION_LIMIT = get_proteomics_results(
    proteomics_diff_csv=DATA / "Proteomics_DiffAnalysis_2WayAnova_.csv",
    proteomics_poi_csv=DATA / "Proteomics_Proteins_of_interest.csv",
    proteomics_concentration_csv=DATA / "Proteomics_samples_proteinconcentration.csv",
    proteomics_dry_weight_csv=DATA / "Proteomics_Dry_weights.csv",
    proteomics_dilutions_csv=DATA / "Proteomics_samples_dilutions.csv",
    proteomics_od_wt_csv=DATA / "20240328_Spectrum_OD_WT_proteomics.csv",
    proteomics_od_sqs_csv=DATA / "20240329_Spectrum_OD_sqs_proteomics.csv",
    proteomics_od_dcas9_csv=DATA / "20240329_Spectrum_OD_dCas9_proteomics.csv",
    cell_volume=CELL_VOLUME,
    protein_content=PROTEIN_CONTENT,
)

ENZYME_CONCENTRATIONS_UMOL: dict[str, float] = get_enzyme_concentrations(
    equilibrium_constants_csv=DATA / "Equilibrium_constants.csv",
    abundances=ABUNDANCES,
    detection_limit=DETECTION_LIMIT,
)

ENZYME_PARAMETERS: dict[str, float] = get_enzyme_parameters(
    mode=MODE,
    blasted_parameters_csv=DATA / "enzyme_parameters_blasted.csv",
    predicted_kms_csv=DATA / "predicted_Kms.csv",
    equilibrium_constants_csv=DATA / "Equilibrium_constants.csv",
)

MIN_KCAT_VALUES: dict[str, float] = calculate_min_kcats(
    enzyme_concentrations_umol=ENZYME_CONCENTRATIONS_UMOL,
    cell_volume=CELL_VOLUME,
    doubling_time=DOUBLING_TIME,
)


LIGHTS = pd.DataFrame(
    {
        path.stem: pd.read_csv(path, index_col=0).squeeze(1)
        for path in sorted((DATA / "lights").glob("*.csv"))
    }
)
EXP_GROWTH_RATE_BY_DENSITY = pd.read_csv(
    DATA / "growth_rates_by_density.csv",
    index_col=0,
)["growth_rate"]

PROTEINS_OF_INTEREST = pd.read_csv(
    DATA / "Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)


def get_parameterised_model(
    strain: Literal["wt", "dshc"],
    growth_mode: Literal["steady", "dynamic"],
    kcat_values: dict[str, float] = MIN_KCAT_VALUES,
    cell_volume: float = CELL_VOLUME,
    enzyme_costs: pd.DataFrame = ENZYME_COSTS,
    enzyme_concentrations_umol: dict[str, float] = ENZYME_CONCENTRATIONS_UMOL,
    enzyme_parameters: dict[str, float] = ENZYME_PARAMETERS,
    biomass_regression_model: Pipeline = BIOMASS_REGRESSION_MODEL,
    abs_coef: pd.DataFrame = ABS_COEF,
    lights: pd.DataFrame = LIGHTS,
    ocp_absorption: pd.DataFrame = OCP_ABSORPTION,
    molar_masses: pd.Series = MOLAR_MASSES,
    ps_comp: pd.DataFrame = PS_COMP,
) -> tuple[Model, dict[str, float]]:
    m = get_model(
        growth_mode=growth_mode,
        strain=strain,
        cell_volume=cell_volume,
        enzyme_total_cost_mol=enzyme_costs,
        enzyme_concentrations_umol=enzyme_concentrations_umol,
        enzyme_parameters=enzyme_parameters,
        biomass_regression_model=biomass_regression_model,
        abs_coef=abs_coef,
        lights=lights,
        ocp_absorption=ocp_absorption,
        molar_masses=molar_masses,
        ps_comp=ps_comp,
    )
    m.update_parameters(
        {
            # "ShcKcat": 0,  # default 600
            # "Shc_ksynth": 0, default 0.008034116888921783
            "light": 100,
            "ATP_out_metabolism": 0.5,
            "3PGA_out_metabolism": 0.5,
            "NADPH_out_metabolism": 0.5,
            "Fdred_out_metabolism": 0.5,
            "cell_volume": cell_volume,
            "DXS_inh_str": 0.001,
            "ThiGDXPKm": 0.01,
        }
    )
    m.update_parameters(
        {k: v for k, v in kcat_values.items() if k not in {"IspHKcat", "CrtEKcat"}}
    )

    # Update initial
    starting_conc = {}
    for i in m.compounds:
        starting_conc[i] = 0.01
    starting_conc.update(enzyme_concentrations_umol)
    starting_conc.update(
        {
            "3PGA": 0.1,
            "ATP": 0.1,
            "NADPH": 0.1,
            "Fdred": 0.1,
            "ThiDP": 10,
            "chlorophyll-phytol": CHLA / cell_volume,
            "Cell_density": 1e1,
            "carotenoids": CAROT / cell_volume,
            "SQ": SQUALENE / cell_volume,
        }
    )
    return m, starting_conc


def get_doubling_time(rel_growth_rate: float) -> float:
    return np.log(2) / (rel_growth_rate * 60)


def generate_initial_conditions() -> dict[str, float]:
    ms, y0 = get_parameterised_model(
        strain="dshc",
        growth_mode="steady",
    )
    return unwrap(
        Simulator(ms)
        # FIXME: explain why this number
        .initialise(y0 | {"Cell_density": EXP_GROWTH_RATE_BY_DENSITY.index[24] / 1e6})
        .simulate_to_steady_state_and()
        .get_new_y0()
    )


Y0_SS = generate_initial_conditions()

In [ ]:
ms, y0 = get_parameterised_model(
    strain="dshc",
    growth_mode="steady",
)
len(ms.parameters)

In [ ]:
# 3 µM/min 3PGA at OD=1, growth rate 0.001 == doubling time around 11 h.
Start_OD = 1
Growth_rate = 0.001  # min^-1
print(np.log(2) / 0.001 / 60)
print(
    "g/L/min calculated from growth: ", Start_OD * (Growth_rate) / 3 / 2
)  # OD*Growth_rate min^-1--> OD/min / g/L CDW/OD --> g/L CDW/min * gC/gCDW --> g/L/min
print(
    "g/L/min calculated from influx prediction:",
    14 * 2e-15 * 1e11 * 186.06 * (36 / 186.06) * 1e-3,
)  # mmol/L/min 3PGA * L/cell--> mmol/cell/min *cells/L --> mmol/L/min * mg/mmol -> mg 3PGA /L/min --> mg C/L/min --> g/L/min
# 0.06 fg/cell s^-1
print(
    "fg/cell/s to compare to paper:", 2.4 * 2e-15 * 186.061 * 1e9 / 60
)  # mmol/L/min --> mmol/cell/min --> mg/cell/min --> fg/cell/min -_> fg/cell/s

print(
    "g/L/min calculated from fg/cell/s value in Paper:", 0.06 * 1e11 * 60 / 1e15
)  # fg/cell/s * cells/L --> fg/L/s --> fg/L/min --> g/L/min

## Fig ??: compare min kcat with literature

In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
        "axes.grid": True,
    }
):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.set_axisbelow(b=True)

    kcat_comp = pd.DataFrame.from_dict(MIN_KCAT_VALUES, "index")
    for key, value in ENZYME_PARAMETERS.items():
        if key in kcat_comp.index:
            kcat_comp.at[key, "A"] = value
    kcat_comp.plot(kind="bar", ax=ax)

    ax.legend([r"Calculated min $\mathrm{k_{cat}}$", r"Literature $\mathrm{k_{cat}}$"])

## Fig 09: growth rate over time

FIXME: you are comparing to experimental results here. Briefly explain experimental details here:

- light was set to 80, because ...
- sample depth was 0.02 m, because ...
- why is ShcKcat different?

In [ ]:
def make_fig_9_data(y0_ss: dict[str, float]) -> pd.DataFrame:
    """If you dislike the markdown stuff, you can also explain it here."""
    md, _ = get_parameterised_model(
        strain="dshc",
        growth_mode="dynamic",
    )
    md.update_parameters(
        {
            "light": 80,  # FIXME: why is this changed?
            "sample_depth_m": 0.0135,  # FIXME: why is this changed?
            "ShcKcat": 0,  # FIXME: why is this changed?
            "Shc_ksynth": 0,  # FIXME: why is this changed?
            "DXS_inh_str": 0,  # FIXME: why is this changed?
        }
    )

    return unwrap(
        Simulator(md)
        # FIXME: explain why this number
        .initialise(y0_ss | {"Cell_density": 54562873.0217531 / 1e6})
        .simulate_and(96 * 60)
        .get_full_results_df()
    )


res_fig9 = make_fig_9_data(Y0_SS)

In [ ]:
def fig9(res: pd.DataFrame) -> None:
    with plt.rc_context(
        {
            "lines.linewidth": 3,
            "font.size": 12.0,
        }
    ):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")

        ax1.plot(res["Growth_rate"] * 60)
        ax1.scatter(
            x=np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY)),
            y=EXP_GROWTH_RATE_BY_DENSITY,
            c="C1",
        )
        ax1.set_ylabel("Growth rate $[h^{-1}]$")
        ax1.set_xlabel("Time [h]")
        ax1.set_xticks(np.arange(0, 97 * 60, 24 * 60))
        ax1.set_xticklabels(np.arange(0, 97, 24))
        ax1.grid()

        ax2.plot(
            res["Cell_density"] * 1e6,
            label="Simulated values",
        )
        ax2.plot(
            np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY[::2])),
            EXP_GROWTH_RATE_BY_DENSITY[::2].index,
            label="Experimental values",
        )
        ax2.set_yscale("log")
        ax2.set_ylabel(r"Cell density $[Cells\ mL^{-1}]$")
        ax2.set_xlabel("Time [h]")
        ax2.set_xticks(np.arange(0, 97 * 60, 24 * 60))
        ax2.set_xticklabels(np.arange(0, 97, 24))
        ax2.grid()

        ax2.legend(loc="lower right")
        savefig(fig, "fig9-growth.png")
        plt.show()


fig9(res_fig9)

## Fig 10: scan ss growth rate by cell density

FIXME: explain experimental details here:

- light was set to 80, because ...
- sample depth was 0.02 m, because ...

In [ ]:
def _worker(
    density: float,
    model: Model,
    y0: dict[str, float],
) -> float | None:
    try:
        df = (
            Simulator(model)
            .initialise(y0 | {"Cell_density": density / 1e6})
            .simulate_to_steady_state_and()
            .get_full_results_df()
        )
        if df is not None:
            return df.iloc[-1]["Growth_rate"] * 60
    except:  # noqa: E722
        print(f"Failed for density={density}")
    return None


def make_fig10_data() -> pd.Series:
    # FIXME: any reason we are not using the Y0_SS here?
    ms, y0 = get_parameterised_model(
        strain="dshc",
        growth_mode="steady",
    )
    ms.update_parameters(
        {
            "light": 80,  # FIXME: why is this changed?
            "sample_depth_m": 0.02,  # FIXME: why is this changed?
        }
    )

    return pd.Series(
        parallelise(
            partial(_worker, model=ms, y0=y0),
            inputs=list(
                zip(
                    EXP_GROWTH_RATE_BY_DENSITY.index[::2],
                    EXP_GROWTH_RATE_BY_DENSITY.index[::2],
                    strict=True,
                )
            ),
        ),
        dtype=float,
    )


growth_rate_by_density = make_fig10_data()


In [ ]:
def fig10(growth_rate_by_density: pd.Series) -> None:
    with plt.rc_context(
        {
            "lines.linewidth": 3,
            "font.size": 12.0,
            "axes.grid": True,
        }
    ):
        fig, ax = plt.subplots(figsize=(4, 3))
        growth_rate_by_density.plot(ax=ax, label="Simulation")
        EXP_GROWTH_RATE_BY_DENSITY.plot(ax=ax, marker="o", label="Experimental")
        ax.set_xscale("log")
        ax.set_ylim(0)
        ax.set_ylabel(r"Growth rate [$h^{-1}$]")
        ax.set_xlabel(r"Cell density [$Cells\ ml^{-1}$]")
        ax.legend()
        savefig(fig, "fig10-growth-by-density.png")
        plt.show()


fig10(growth_rate_by_density)

## Fig 11: Growth repression induction time point tests

FIXME: explain

Unmodified dshc strain

In [ ]:
def repress_crte(s: _Simulate) -> _Simulate:
    s.update_parameter("CrtE3Kcat", 0.01)
    return s


def product_after_induction(
    time_point_of_induction: float,
    t_end: float,
    y0_ss: dict[str, float],
) -> pd.Series:
    md, _ = get_parameterised_model(
        strain="dshc",
        growth_mode="dynamic",
    )

    s = Simulator(md).initialise(y0_ss).simulate_and(time_point_of_induction)
    s = repress_crte(s)
    s.simulate(t_end)

    if (c := s.get_full_results_df()) is None:
        return pd.Series({})

    return c.iloc[-1]


def make_fig11_data(y0: dict[str, float]) -> pd.DataFrame:
    fourteen_days_in_min = 14 * DAY_IN_MIN
    time_points_of_induction = np.arange(
        0,
        fourteen_days_in_min,
        DAY_IN_MIN / 2,
        dtype=float,
    )

    return pd.DataFrame(
        parallelise(
            partial(product_after_induction, y0_ss=y0, t_end=fourteen_days_in_min),
            list(zip(time_points_of_induction, time_points_of_induction, strict=True)),
        )
    ).T


res_fig11 = make_fig11_data(y0=Y0_SS)

In [ ]:
def fig11(res: pd.DataFrame) -> None:
    with plt.rc_context(
        {
            "lines.linewidth": 3,
            "font.size": 12.0,
            "axes.grid": True,
        }
    ):
        fig, (ax1, ax2, ax3) = plt.subplots(
            1,
            3,
            figsize=(8, 3),
            layout="constrained",
        )

        fig.suptitle("Total concentration after 14 days")

        # FIXME: explain magic numbers
        ax1.plot(
            res.index / DAY_IN_MIN,
            res["Cell_density"] * 1e9 * res["SQ"] * 410.73 * CELL_VOLUME,
        )
        ax1.set_ylabel(
            "Squalene conc. in culture\n"
            r"[$mg\ L^{-1}$]"
        )

        ax2.plot(
            res.index / DAY_IN_MIN,
            res["Cell_density"] * 1e9,
        )
        ax2.set_yscale("log")
        ax2.set_ylabel(r"Cell density [$Cells\ L^{-1}$]")

        # FIXME: explain magic numbers
        ax3.plot(
            res.index / DAY_IN_MIN,
            res["SQ"] * 410.73 / 1000,
        )
        ax3.set_ylabel(
            "Intracellular squalene\n"
            r"concentration [$g\ L^{-1}$]"
        )

        for ax in (ax1, ax2, ax3):
            ax.set_xlabel("Timepoint of induction [d]")
        savefig(fig, "fig11-induction.png")
        plt.show()


fig11(res_fig11)


## Fig 12: Gradient descent MCA, with no repeat overexpression allowed

FIXME: quickly explain the approach

- standard conditions

In [ ]:
def make_fig12_data(
    y0: dict[str, float], poi: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    ms, y0 = get_parameterised_model(
        strain="dshc",
        growth_mode="steady",
    )

    pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]

    frcs_by_run = {}
    fluxes_by_run = {}
    overexpressed = []  # To keep track of modified parameters in order

    for runs in np.arange(0, 10):
        # Get flux response coefficients
        _, frc = mca.get_response_coefficients_df(ms, pars_for_mca, y0)
        frcs_by_run[runs] = frc

        # Get steady-state fluxes for the given run
        s = Simulator(ms).initialise(y0).simulate_to_steady_state_and()
        fluxes_by_run[runs] = s.get_fluxes_df().iloc[-1]

        if (yss := s.get_new_y0()) is None:
            break
        else:
            y0 = yss

        # Filter out already modified parameters
        frc_filtered = frc.drop(index=overexpressed)
        most_positive = cast(
            str,
            (frc_filtered["Sqs"] + frc_filtered["Biomass"]).idxmax(),
        )

        # It's possible that none was found
        best = poi[poi["Enzyme"] == most_positive[:-7]]["kDa"]
        if len(best) == 0:
            break

        # FIXME: explain this factor
        overexpression_amount = 1.331057 * best.iloc[0]
        ms.update_parameters(
            {
                most_positive: ms.get_parameter(most_positive)
                + overexpression_amount / (5.5 * 60)  # FIXME: explain factor
            }
        )
        overexpressed.append(most_positive)
    return (
        pd.concat(frcs_by_run),
        cast(pd.DataFrame, pd.concat(fluxes_by_run).unstack()),
        overexpressed,
    )


frc_by_run, fluxes_by_run, overexpressed = make_fig12_data(
    y0=Y0_SS,
    poi=PROTEINS_OF_INTEREST,
)


In [ ]:
def fig12(
    frc_by_run: pd.DataFrame, fluxes_by_run: pd.DataFrame, overexpressed: list[str]
):
    with plt.rc_context(
        {
            "lines.linewidth": 3,
            # "font.size": 12.0,
        }
    ):
        fig, (ax1, ax2) = plt.subplots(
            2,
            1,
            figsize=(7, 5),
            layout="constrained",
            height_ratios=(5, 1),
            sharex=True,
        )

        # Create the heatmap
        # FIXME: explain the min.abs logic here
        data = (
            frc_by_run["Sqs"].unstack().T
            + frc_by_run["Biomass"].unstack().min(axis=1).abs()
        )
        sns.heatmap(
            data,
            annot=False,
            fmt=".2f",
            cmap="RdBu_r",
            center=0,
            ax=ax1,
        )
        ax1.set_ylabel("Adjustable parameters", fontsize=12)
        # Draw rectangles around the overexpressed genes
        for run, gene in enumerate(overexpressed):
            ax1.add_patch(
                plt.Rectangle(
                    (run, data.index.get_loc(gene)),
                    1,
                    1,
                    fill=False,
                    edgecolor="red",
                    linewidth=3,
                )
            )

        # Line plot
        # FIXME: explain factor
        ax2.plot(fluxes_by_run["Sqs"] / fluxes_by_run["Biomass"] * 410.73 / 1000)
        ax2.set_ylim(0)
        ax2.grid(True)
        ax2.set_xlabel("No. of genes overexpressed")
        ax2.set_ylabel(
            "Intracellular squalene \n"
            r"concentration [$g\ L^{-1}$]",
            rotation=0,
        )
        ax2.yaxis.set_label_coords(-0.35, 0.15)
        savefig(fig, "fig12-overexpression.png")
        plt.show()


fig12(frc_by_run, fluxes_by_run, overexpressed)